# Gold — Construção do Modelo Analítico

Este notebook contempla a construção da camada Gold a partir dos dados tratados e validados na camada Silver.

Nesta etapa, os dados são integrados e organizados conforme o modelo dimensional definido para o projeto, resultando nas tabelas fato e dimensão utilizadas nas análises de negócio.

## 01. Preparação da Camada Gold

A camada Gold é destinada ao armazenamento das tabelas do modelo analítico, construídas a partir dos dados previamente tratados na camada Silver.

Nesta etapa é criado o schema utilizado para persistência das tabelas fato e dimensão do projeto.

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.gold;

## 02. Construção da Dimensão Tempo

A dimensão `dm_tempo` é construída a partir do período existente nos dados de compra dos pedidos e contém uma linha para cada data compreendida entre a primeira e a última compra registrada.

A dimensão permite a realização de análises temporais por dia, mês, trimestre e ano. O campo `data` é utilizado para relacionamento com a data de compra presente nas tabelas fato.

### 02.1 Criação da Dimensão

In [0]:
%sql
-- Cria a dimensão de tempo considerando todo o período de compras disponível na Silver.

CREATE OR REPLACE TABLE workspace.gold.dm_tempo
USING DELTA
AS

WITH periodo AS (
    SELECT
        MIN(CAST(order_purchase_timestamp AS DATE)) AS data_minima,
        MAX(CAST(order_purchase_timestamp AS DATE)) AS data_maxima
    FROM workspace.silver.orders
),

calendario AS (
    SELECT
        EXPLODE(
            SEQUENCE(
                data_minima,
                data_maxima,
                INTERVAL 1 DAY
            )
        ) AS data
    FROM periodo
)

SELECT
    data,
    DAY(data) AS dia,
    MONTH(data) AS mes,

    CASE MONTH(data)
        WHEN 1 THEN 'Janeiro'
        WHEN 2 THEN 'Fevereiro'
        WHEN 3 THEN 'Março'
        WHEN 4 THEN 'Abril'
        WHEN 5 THEN 'Maio'
        WHEN 6 THEN 'Junho'
        WHEN 7 THEN 'Julho'
        WHEN 8 THEN 'Agosto'
        WHEN 9 THEN 'Setembro'
        WHEN 10 THEN 'Outubro'
        WHEN 11 THEN 'Novembro'
        WHEN 12 THEN 'Dezembro'
    END AS nome_mes,

    QUARTER(data) AS trimestre,
    YEAR(data) AS ano

FROM calendario;

In [0]:
%sql

select * from gold.dm_tempo

### 02.2 Validação

Após a persistência da dimensão, são realizadas validações para verificar o período contemplado e garantir a unicidade do campo `data`.

In [0]:
%sql
-- Valida o período, o volume e a unicidade das datas da dimensão de tempo.

SELECT
    MIN(data) AS data_minima,
    MAX(data) AS data_maxima,
    COUNT(*) AS qtd_datas,
    COUNT(DISTINCT data) AS qtd_datas_unicas,
    COUNT(*) - COUNT(DISTINCT data) AS qtd_duplicados
FROM workspace.gold.dm_tempo;

## 03. Construção da Dimensão Produto

A dimensão `dm_produto` é construída a partir da tabela `silver.products`, que já contém os dados de produtos tratados e enriquecidos com as informações de categoria.

Nesta etapa, os campos são selecionados e renomeados para a nomenclatura adotada no modelo analítico, mantendo a granularidade de uma linha por produto.

### 03.1 Criação da Dimensão

In [0]:
%sql
-- Cria a dimensão de produtos a partir dos dados tratados na camada Silver.

CREATE OR REPLACE TABLE workspace.gold.dm_produto
USING DELTA
AS

SELECT
    product_id AS id_produto,
    product_category_name AS categoria_produto,
    product_category_name_english AS categoria_produto_ingles,
    product_name_length AS tamanho_nome_produto,
    product_description_length AS tamanho_descricao_produto,
    product_photos_qty AS qtd_fotos_produto,
    product_weight_g AS peso_produto_g,
    product_length_cm AS comprimento_produto_cm,
    product_height_cm AS altura_produto_cm,
    product_width_cm AS largura_produto_cm

FROM workspace.silver.products;

In [0]:
%sql

select * from gold.dm_produto

### 03.2 Validação

Após a criação da dimensão, são realizadas validações para garantir a manutenção da granularidade de uma linha por produto e a preservação dos tratamentos de categoria realizados na camada Silver.

In [0]:
%sql
-- Valida o volume e a unicidade de id_produto na dimensão.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT id_produto) AS qtd_produtos,
    COUNT(*) - COUNT(DISTINCT id_produto) AS qtd_duplicados
FROM workspace.gold.dm_produto;

In [0]:
%sql
-- Valida a preservação das categorias tratadas durante a construção da dimensão.

SELECT
    SUM(CASE WHEN categoria_produto IS NULL THEN 1 ELSE 0 END) AS categorias_nulas,
    SUM(CASE WHEN categoria_produto_ingles IS NULL THEN 1 ELSE 0 END) AS traducoes_nulas,
    SUM(CASE WHEN categoria_produto = 'sem_categoria' THEN 1 ELSE 0 END) AS sem_categoria
FROM workspace.gold.dm_produto;

## 04. Construção da Fato Pagamentos

A tabela `ft_pagamentos` é construída a partir dos registros tratados em `silver.order_payments`.

A granularidade definida para a tabela é de uma linha por combinação de pedido e sequência de pagamento, permitindo preservar situações em que um mesmo pedido possui mais de um registro ou forma de pagamento.

Nesta etapa também é aplicada a nomenclatura em português adotada para o modelo analítico.

### 04.1 Criação da Tabela Fato

In [0]:
%sql
-- Cria a fato de pagamentos preservando a granularidade de um registro por sequência de pagamento do pedido.

CREATE OR REPLACE TABLE workspace.gold.ft_pagamentos
USING DELTA
AS

SELECT
    order_id AS id_pedido,
    payment_sequential AS sequencial_pagamento,
    payment_type AS tipo_pagamento,
    payment_installments AS qtd_parcelas,
    payment_value AS valor_pagamento

FROM workspace.silver.order_payments;

In [0]:
%sql

select * from gold.ft_pagamentos order by id_pedido, sequencial_pagamento

### 04.2 Validação

Após a persistência da tabela fato, são realizadas validações para garantir a manutenção da granularidade dos registros e a preservação dos valores e características provenientes da camada Silver.

In [0]:
%sql
-- Valida o volume e a granularidade de id_pedido + sequencial_pagamento na fato de pagamentos.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(
        DISTINCT CONCAT(id_pedido, '|', sequencial_pagamento)
    ) AS qtd_pagamentos_unicos,
    COUNT(*) - COUNT(
        DISTINCT CONCAT(id_pedido, '|', sequencial_pagamento)
    ) AS qtd_duplicados
FROM workspace.gold.ft_pagamentos;

In [0]:
%sql
-- Valida se os valores monetários e os casos atípicos tratados na Silver foram preservados na Gold.

SELECT
    SUM(CASE WHEN qtd_parcelas = 0 THEN 1 ELSE 0 END) AS parcelas_zero,
    SUM(CASE WHEN tipo_pagamento = 'not_defined' THEN 1 ELSE 0 END) AS tipo_nao_definido,
    SUM(CASE WHEN valor_pagamento = 0 THEN 1 ELSE 0 END) AS valor_zero,
    MIN(valor_pagamento) AS valor_minimo,
    MAX(valor_pagamento) AS valor_maximo
FROM workspace.gold.ft_pagamentos;

## 05. Construção da Fato Vendas

A tabela `ft_vendas` representa os itens comercializados nos pedidos e possui granularidade de uma linha por combinação de pedido e item do pedido.

Sua construção parte de `silver.order_items` e incorpora informações de `silver.orders` e `silver.customers` necessárias às análises, mantendo os identificadores de produto e vendedor para relacionamento e detalhamento dos registros.

O identificador único do cliente (`id_cliente_unico`) é utilizado para permitir análises de comportamento e recompra, enquanto `data_compra` possibilita o relacionamento com a dimensão `dm_tempo`.

Os valores do item e do frete são preservados individualmente nessa granularidade, evitando agregações antecipadas dos registros de venda.

### 05.1 Criação da Tabela Fato

A construção combina os itens de pedido com os dados do pedido e do cliente. Como `orders` possui uma linha por pedido e `customers` uma linha por `customer_id`, os relacionamentos adicionam atributos à venda sem alterar sua granularidade original.

In [0]:
%sql
-- Cria a fato de vendas na granularidade de um registro por item do pedido.

CREATE OR REPLACE TABLE workspace.gold.ft_vendas
USING DELTA
AS

SELECT
    oi.order_id AS id_pedido,
    CAST(oi.order_item_id AS INT) AS id_item_pedido,
    oi.product_id AS id_produto,
    oi.seller_id AS id_vendedor,
    c.customer_unique_id AS id_cliente_unico,
    CAST(o.order_purchase_timestamp AS DATE) AS data_compra,
    oi.price AS valor_item,
    oi.freight_value AS valor_frete_item

FROM workspace.silver.order_items oi

LEFT JOIN workspace.silver.orders o
    ON oi.order_id = o.order_id

LEFT JOIN workspace.silver.customers c
    ON o.customer_id = c.customer_id;

In [0]:
%sql

select * from gold.ft_vendas

### 05.2 Validação

Após a persistência da tabela, são realizadas validações para confirmar a manutenção da granularidade de uma linha por item do pedido, verificar se os relacionamentos utilizados na construção não provocaram perda ou multiplicação de registros e avaliar a integridade das referências às dimensões.

In [0]:
%sql
-- Valida o volume e a granularidade de id_pedido + id_item_pedido na fato de vendas.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT CONCAT(id_pedido, '|', id_item_pedido)) AS qtd_itens_unicos,
    COUNT(*) - COUNT(DISTINCT CONCAT(id_pedido, '|', id_item_pedido)) AS qtd_duplicados
FROM workspace.gold.ft_vendas;

In [0]:
%sql
-- Valida se a construção da fato preservou todos os itens existentes na camada Silver.

SELECT
    (SELECT COUNT(*) FROM workspace.silver.order_items) AS registros_silver,
    (SELECT COUNT(*) FROM workspace.gold.ft_vendas) AS registros_gold,
    (SELECT COUNT(*) FROM workspace.silver.order_items)
        - (SELECT COUNT(*) FROM workspace.gold.ft_vendas) AS diferenca;

In [0]:
%sql
-- Valida a integridade das referências de produto e data utilizadas pela fato de vendas.

SELECT
    SUM(CASE WHEN p.id_produto IS NULL THEN 1 ELSE 0 END) AS produtos_sem_dimensao,
    SUM(CASE WHEN t.data IS NULL THEN 1 ELSE 0 END) AS datas_sem_dimensao
FROM workspace.gold.ft_vendas v
LEFT JOIN workspace.gold.dm_produto p
    ON v.id_produto = p.id_produto
LEFT JOIN workspace.gold.dm_tempo t
    ON v.data_compra = t.data;

## 06. Construção da Fato Pedidos

A tabela `ft_pedidos` representa o evento de pedido e possui granularidade de uma linha por `id_pedido`.

Sua construção parte de `silver.orders` e incorpora os atributos do cliente presentes em `silver.customers`, além de métricas consolidadas provenientes de `silver.order_items` e `silver.order_reviews`.

Como itens e avaliações podem possuir múltiplos registros para um mesmo pedido, essas fontes são previamente agregadas na granularidade de pedido antes da realização dos relacionamentos. Essa estratégia evita a multiplicação de registros e valores durante os joins.

A tabela resultante concentra informações relacionadas ao ciclo do pedido, cliente, valores comercializados e avaliações, servindo como principal fonte para análises realizadas no nível de pedido.

### 06.1 Criação da Tabela Fato

Os itens são agregados por pedido para obtenção da quantidade de itens, valor dos produtos, valor do frete e valor total do pedido.

As avaliações também são consolidadas por pedido, utilizando a quantidade de avaliações e a nota média. Pedidos sem avaliação permanecem no modelo com `qtd_avaliacoes = 0` e `nota_media_avaliacao` nula, evitando a atribuição artificial de uma nota.

Após as agregações, as informações são relacionadas à tabela de pedidos e aos dados de clientes, preservando todos os pedidos existentes na fonte.__

In [0]:
%sql
-- Cria a fato de pedidos consolidando itens e avaliações na granularidade de um registro por pedido.

CREATE OR REPLACE TABLE workspace.gold.ft_pedidos
USING DELTA
AS

WITH itens_por_pedido AS (

    SELECT
        order_id,
        COUNT(*) AS qtd_itens,
        SUM(price) AS valor_total_produtos,
        SUM(freight_value) AS valor_total_frete,
        SUM(price + freight_value) AS valor_total_pedido
    FROM workspace.silver.order_items
    GROUP BY order_id

),

avaliacoes_por_pedido AS (

    SELECT
        order_id,
        COUNT(*) AS qtd_avaliacoes,
        AVG(review_score) AS nota_media_avaliacao
    FROM workspace.silver.order_reviews
    GROUP BY order_id

)

SELECT
    o.order_id AS id_pedido,
    o.customer_id AS id_cliente,
    c.customer_unique_id AS id_cliente_unico,
    c.customer_city AS cidade_cliente,
    c.customer_state AS uf_cliente,

    CAST(o.order_purchase_timestamp AS DATE) AS data_compra,
    o.order_status AS status_pedido,

    o.order_purchase_timestamp AS data_hora_compra,
    o.order_approved_at AS data_hora_aprovacao,
    o.order_delivered_carrier_date AS data_hora_envio_transportadora,
    o.order_delivered_customer_date AS data_hora_entrega_cliente,
    CAST(o.order_estimated_delivery_date AS DATE) AS data_estimada_entrega,

    COALESCE(i.qtd_itens, 0) AS qtd_itens,
    COALESCE(i.valor_total_produtos, CAST(0 AS DECIMAL(10,2))) AS valor_produtos,
    COALESCE(i.valor_total_frete, CAST(0 AS DECIMAL(10,2))) AS valor_frete,
    COALESCE(i.valor_total_pedido, CAST(0 AS DECIMAL(10,2))) AS valor_total_pedido,

    COALESCE(a.qtd_avaliacoes, 0) AS qtd_avaliacoes,
    a.nota_media_avaliacao AS nota_media_avaliacao

FROM workspace.silver.orders o

LEFT JOIN workspace.silver.customers c
    ON o.customer_id = c.customer_id

LEFT JOIN itens_por_pedido i
    ON o.order_id = i.order_id

LEFT JOIN avaliacoes_por_pedido a
    ON o.order_id = a.order_id;

In [0]:
%sql

select * from gold.ft_pedidos

### 06.2 Validação

Após a construção da tabela, são realizadas validações para garantir a preservação da granularidade de uma linha por pedido, a manutenção dos pedidos existentes na camada Silver e a consistência das métricas obtidas a partir das agregações de itens e avaliações.

In [0]:
%sql
-- Valida o volume e a unicidade de id_pedido na fato de pedidos.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT id_pedido) AS qtd_pedidos_unicos,
    COUNT(*) - COUNT(DISTINCT id_pedido) AS qtd_duplicados
FROM workspace.gold.ft_pedidos;

In [0]:
%sql
-- Valida se todos os pedidos existentes na Silver foram preservados na fato de pedidos.

SELECT
    (SELECT COUNT(*) FROM workspace.silver.orders) AS pedidos_silver,
    (SELECT COUNT(*) FROM workspace.gold.ft_pedidos) AS pedidos_gold,
    (SELECT COUNT(*) FROM workspace.silver.orders)
        - (SELECT COUNT(*) FROM workspace.gold.ft_pedidos) AS diferenca;

In [0]:
%sql
-- Quantifica pedidos sem itens e sem avaliações após as consolidações realizadas na fato.

SELECT
    SUM(CASE WHEN qtd_itens = 0 THEN 1 ELSE 0 END) AS pedidos_sem_itens,
    SUM(CASE WHEN qtd_avaliacoes = 0 THEN 1 ELSE 0 END) AS pedidos_sem_avaliacao,
    SUM(CASE WHEN qtd_avaliacoes = 0 AND nota_media_avaliacao IS NOT NULL THEN 1 ELSE 0 END) AS inconsistencias_avaliacao
FROM workspace.gold.ft_pedidos;

In [0]:
%sql
-- Valida se todas as datas de compra da fato de pedidos possuem correspondência na dimensão de tempo.

SELECT
    COUNT(*) AS datas_sem_dimensao
FROM workspace.gold.ft_pedidos p
LEFT JOIN workspace.gold.dm_tempo t
    ON p.data_compra = t.data
WHERE t.data IS NULL;

In [0]:
%sql
-- Compara os valores totais de produtos e frete entre ft_vendas e ft_pedidos.

SELECT
    v.valor_produtos_vendas,
    p.valor_produtos_pedidos,
    v.valor_produtos_vendas - p.valor_produtos_pedidos AS diferenca_produtos,

    v.valor_frete_vendas,
    p.valor_frete_pedidos,
    v.valor_frete_vendas - p.valor_frete_pedidos AS diferenca_frete,

    v.valor_total_vendas,
    p.valor_total_pedidos,
    v.valor_total_vendas - p.valor_total_pedidos AS diferenca_total

FROM (
    SELECT
        SUM(valor_item) AS valor_produtos_vendas,
        SUM(valor_frete_item) AS valor_frete_vendas,
        SUM(valor_item + valor_frete_item) AS valor_total_vendas
    FROM workspace.gold.ft_vendas
) v

CROSS JOIN (
    SELECT
        SUM(valor_produtos) AS valor_produtos_pedidos,
        SUM(valor_frete) AS valor_frete_pedidos,
        SUM(valor_total_pedido) AS valor_total_pedidos
    FROM workspace.gold.ft_pedidos
) p;


In [0]:
%sql
-- Valida pedido a pedido se os valores agregados da ft_vendas correspondem aos valores da ft_pedidos.

WITH vendas_por_pedido AS (
    SELECT
        id_pedido,
        SUM(valor_item) AS valor_produtos_vendas,
        SUM(valor_frete_item) AS valor_frete_vendas,
        SUM(valor_item + valor_frete_item) AS valor_total_vendas
    FROM workspace.gold.ft_vendas
    GROUP BY id_pedido
)

SELECT
    COUNT(*) AS pedidos_com_divergencia
FROM workspace.gold.ft_pedidos p

INNER JOIN vendas_por_pedido v
    ON p.id_pedido = v.id_pedido

WHERE
    p.valor_produtos <> v.valor_produtos_vendas
    OR p.valor_frete <> v.valor_frete_vendas
    OR p.valor_total_pedido <> v.valor_total_vendas;

## 07. Validação Geral da Camada Gold

Após a construção das tabelas fato e dimensão, foi realizada uma validação consolidada da camada Gold.

As verificações realizadas ao longo da construção contemplaram a granularidade e unicidade das chaves definidas para cada tabela, a preservação dos registros provenientes da camada Silver, a integridade das referências às dimensões e a consistência dos valores entre as diferentes granularidades do modelo.

A validação entre `ft_vendas` e `ft_pedidos` também confirmou que os valores de produtos e frete consolidados no nível de pedido correspondem aos valores registrados no nível de item.

In [0]:
%sql
-- Apresenta as tabelas Gold e seus volumes após a conclusão da construção e das validações.

SELECT 'dm_produto' AS tabela, COUNT(*) AS qtd_registros FROM workspace.gold.dm_produto
UNION ALL
SELECT 'dm_tempo', COUNT(*) FROM workspace.gold.dm_tempo
UNION ALL
SELECT 'ft_pagamentos', COUNT(*) FROM workspace.gold.ft_pagamentos
UNION ALL
SELECT 'ft_vendas', COUNT(*) FROM workspace.gold.ft_vendas
UNION ALL
SELECT 'ft_pedidos', COUNT(*) FROM workspace.gold.ft_pedidos
ORDER BY tabela;

## 08. Catalogo de Dados da Camada Gold

Após a construção das tabelas fato e dimensão, e após validação, vamos incluir as informações de descrição da tabela e campos, tipo, domínio e linhagem na Unit Catalog do Databricks.

In [0]:
%sql
-- Exibe a estrutura física da dimensão de tempo para documentação do catálogo.

DESCRIBE workspace.gold.dm_tempo;

In [0]:
%sql
-- Exibe a estrutura física da dimensão de produto para documentação do catálogo.

DESCRIBE workspace.gold.dm_produto;

In [0]:
%sql
-- Exibe a estrutura física da fato pagamentos para documentação do catálogo.

DESCRIBE workspace.gold.ft_pagamentos;

In [0]:
%sql
-- Exibe a estrutura física da fato de vendas para documentação do catálogo.

DESCRIBE workspace.gold.ft_vendas;

In [0]:
%sql
-- Exibe a estrutura física da fato de pedidos para documentação do catálogo.

DESCRIBE workspace.gold.ft_pedidos;

### 08.1 dm_tempo

In [0]:
%sql
-- Documenta o contexto e a granularidade da dimensão de tempo.

COMMENT ON TABLE workspace.gold.dm_tempo IS
'Dimensão de tempo utilizada para análises temporais de pedidos e vendas. Granularidade: uma linha por data.';

In [0]:
%sql
-- Documenta os campos da dimensão de tempo no catálogo da camada Gold.

ALTER TABLE workspace.gold.dm_tempo ALTER COLUMN data COMMENT
'Data calendário utilizada para relacionamento com as tabelas fato.';

ALTER TABLE workspace.gold.dm_tempo ALTER COLUMN dia COMMENT
'Dia do mês. Domínio: valores inteiros de 1 a 31.';

ALTER TABLE workspace.gold.dm_tempo ALTER COLUMN mes COMMENT
'Número do mês. Domínio: valores inteiros de 1 a 12.';

ALTER TABLE workspace.gold.dm_tempo ALTER COLUMN nome_mes COMMENT
'Nome do mês em português. Domínio: Janeiro a Dezembro.';

ALTER TABLE workspace.gold.dm_tempo ALTER COLUMN trimestre COMMENT
'Trimestre do ano. Domínio: valores inteiros de 1 a 4.';

ALTER TABLE workspace.gold.dm_tempo ALTER COLUMN ano COMMENT
'Ano correspondente à data.';

### 08.2 dm_produto

In [0]:
%sql
-- Documenta o contexto e a granularidade da dimensão de produtos.

COMMENT ON TABLE workspace.gold.dm_produto IS
'Dimensão contendo atributos descritivos e físicos dos produtos. Granularidade: uma linha por produto.';

In [0]:
%sql
-- Documenta os campos transformados e enriquecidos da dimensão de produtos dos campos que não persistiu a descrição da camada anterior.

ALTER TABLE workspace.gold.dm_produto ALTER COLUMN categoria_produto COMMENT
'Categoria do produto em português. Domínio original: 73 categorias distintas. Produtos sem categoria identificada foram classificados como sem_categoria.';

ALTER TABLE workspace.gold.dm_produto ALTER COLUMN categoria_produto_ingles COMMENT
'Categoria do produto em inglês. Obtida pela tradução da categoria em português. Quando não existe tradução disponível, é preservado o nome original da categoria.';

ALTER TABLE workspace.gold.dm_produto ALTER COLUMN tamanho_nome_produto COMMENT
'Quantidade de caracteres do nome do produto. Domínio observado: 5 a 76. Pode apresentar valores nulos.';

ALTER TABLE workspace.gold.dm_produto ALTER COLUMN tamanho_descricao_produto COMMENT
'Quantidade de caracteres da descrição do produto. Domínio observado: 4 a 3.992. Pode apresentar valores nulos.';

ALTER TABLE workspace.gold.dm_produto ALTER COLUMN qtd_fotos_produto COMMENT
'Quantidade de fotos associadas ao produto. Domínio observado: 1 a 20. Pode apresentar valores nulos.';

### 08.3 ft_pagamentos

In [0]:
%sql
-- Documenta o contexto e a granularidade da fato de pagamentos.

COMMENT ON TABLE workspace.gold.ft_pagamentos IS
'Tabela fato contendo os registros de pagamento dos pedidos. Granularidade: uma linha por combinação de pedido e sequência de pagamento.';

In [0]:
%sql
-- Documenta os campos ainda não descritos da fato de pagamentos.

ALTER TABLE workspace.gold.ft_pagamentos ALTER COLUMN sequencial_pagamento COMMENT
'Número sequencial do pagamento dentro do pedido. Domínio observado: 1 a 29. Em conjunto com id_pedido define a granularidade da tabela.';

ALTER TABLE workspace.gold.ft_pagamentos ALTER COLUMN qtd_parcelas COMMENT
'Quantidade de parcelas associadas ao pagamento. Domínio observado: 0 a 24 parcelas. Foram preservados os valores originais da fonte.';

ALTER TABLE workspace.gold.ft_pagamentos ALTER COLUMN valor_pagamento COMMENT
'Valor associado ao registro de pagamento, em reais. Domínio observado: R$ 0,00 a R$ 13.664,08.';

### 08.4 ft_vendas

In [0]:
%sql
-- Documenta o contexto e a granularidade da fato de vendas.

COMMENT ON TABLE workspace.gold.ft_vendas IS
'Tabela fato contendo os itens comercializados nos pedidos. Granularidade: uma linha por combinação de pedido e item do pedido.';

In [0]:
%sql
-- Documenta os campos ainda não descritos da fato de vendas.

ALTER TABLE workspace.gold.ft_vendas ALTER COLUMN id_item_pedido COMMENT
'Número sequencial do item dentro do pedido. Domínio observado: 1 a 21. Em conjunto com id_pedido define a granularidade da tabela.';

ALTER TABLE workspace.gold.ft_vendas ALTER COLUMN data_compra COMMENT
'Data de realização da compra associada ao item. Utilizada para relacionamento com a dimensão dm_tempo.';

ALTER TABLE workspace.gold.ft_vendas ALTER COLUMN valor_item COMMENT
'Valor monetario do produto correspondente ao item do pedid. Domínio observado:  0,85 a 6.735,00.';

ALTER TABLE workspace.gold.ft_vendas ALTER COLUMN valor_frete_item COMMENT
'Valor monetario do frete associado ao item do pedido. Domínio observado: 0,00 a 409,68.';

### 08.5 ft_pedidos

In [0]:
%sql
-- Documenta o contexto e a granularidade da fato de pedidos.

COMMENT ON TABLE workspace.gold.ft_pedidos IS
'Tabela fato contendo informações consolidadas dos pedidos, incluindo dados do cliente, ciclo do pedido, valores dos itens e avaliações. Granularidade: uma linha por pedido.';

In [0]:
%sql
-- Documenta os campos temporais da fato de pedidos.

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN data_compra COMMENT
'Data de realização da compra. Utilizada para relacionamento com a dimensão dm_tempo.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN data_hora_compra COMMENT
'Data e hora de realização do pedido. Não apresenta valores nulos.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN data_hora_aprovacao COMMENT
'Data e hora de aprovação do pedido. Pode apresentar valores nulos conforme os dados disponíveis na fonte.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN data_hora_envio_transportadora COMMENT
'Data e hora em que o pedido foi encaminhado à transportadora. Pode apresentar valores nulos conforme o status do pedido.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN data_hora_entrega_cliente COMMENT
'Data e hora em que o pedido foi entregue ao cliente. Pode apresentar valores nulos conforme o status do pedido.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN data_estimada_entrega COMMENT
'Data estimada para entrega do pedido ao cliente. Não apresenta valores nulos.';

-- Documenta as métricas de itens e valores consolidadas por pedido.

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN qtd_itens COMMENT
'Quantidade de itens associados ao pedido. Derivada da contagem dos itens do pedido. Pedidos sem itens recebem valor 0.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN valor_produtos COMMENT
'Valor monetário total dos produtos do pedido. Derivado da soma dos valores dos itens. Pedidos sem itens recebem valor R$ 0,00.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN valor_frete COMMENT
'Valor monetário total do frete do pedido. Derivado da soma dos valores de frete dos itens. Pedidos sem itens recebem valor R$ 0,00.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN valor_total_pedido COMMENT
'Valor monetário total do pedido, calculado pela soma dos valores dos produtos e do frete.';

-- Documenta as métricas de avaliações consolidadas por pedido.

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN qtd_avaliacoes COMMENT
'Quantidade de avaliações associadas ao pedido. Derivada da contagem dos registros de avaliação. Pedidos sem avaliação recebem valor 0.';

ALTER TABLE workspace.gold.ft_pedidos ALTER COLUMN nota_media_avaliacao COMMENT
'Nota média das avaliações associadas ao pedido. Domínio: 1 a 5. Permanece nula quando o pedido não possui avaliação.';